# RAG System - Document Q&A & Chatbot

This notebook provides a complete RAG system with:
- Hybrid retrieval (BGE + BM25 + FAISS)
- Ollama/gemma4 generation
- Simple query interface
- Interactive chatbot

In [1]:
# Setup and Imports
import sys
import os
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path('.').absolute()))

from src.rag_system import RAGSystem
print('RAG System imported successfully')

W0520 00:03:47.682000 51692 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.



RAG System imported successfully


## Initialize RAG System

In [2]:
# Initialize RAG system with source directory
SOURCE_DIR = Path('sources')

rag = RAGSystem(source_dir=SOURCE_DIR)
print('RAG System initialized')

# Check Ollama and preload model
from src.ollama_manager import OllamaManager

print("\n=== Ollama Status Check ===")
ollama_mgr = OllamaManager(
    base_url=rag.generator.config.base_url,
    model=rag.generator.config.model
)

status = ollama_mgr.get_status_summary()
if status["ollama_available"]:
    print("✅ Ollama is running")
    print(f"Available models: {status['available_models']}")

    if status["matching_model"]:
        print(f"✅ Model '{status['matching_model']}' is available")
        print(f"\n⏳ Loading model into memory...")
        success, elapsed = ollama_mgr.preload_model(status["matching_model"])
        if success:
            print(f"✅ Model ready ({elapsed:.1f}s)")
            rag.generator.config.model = status["matching_model"]
        else:
            print(f"⚠️ Model may not be fully loaded")
    else:
        print(f"⚠️ Configured model '{status['configured_model']}' not found")
else:
    print("❌ Ollama is NOT running")
    print("   Start with: `ollama serve`")

RAG System initialized

=== Ollama Status Check ===
✅ Ollama is running
Available models: ['gemma4:e4b', 'qwen3.5:4b', 'mistrallite:7b', 'mistral-openorca:7b', 'mistral:7b', 'qwen2:7b', 'qwen3:8b', 'qwen2.5:7b', 'llama3:8b', 'llama3.2:3b', 'gemma3:4b', 'gemma:7b', 'gemma2:9b', 'gemini-3-flash-preview:cloud', 'gemini-3-flash-preview:latest', 'llama3.1:8b', 'mistral-nemo:12b']
✅ Model 'gemma4:e4b' is available

⏳ Loading model into memory...
✅ Model ready (149.0s)


## Ingest Documents

In [3]:
# Ingest and index all documents
stats = rag.ingest_documents(SOURCE_DIR)
print(f"Ingested {stats['documents_loaded']} documents")
print(f"Created {stats['chunks_created']} chunks")
print(f"System stats: {rag.get_stats()}")

Loading cached index for 6 files...
Loading vector index from .rag_index\vectors.faiss...
Loading BM25 index from .rag_index\bm25.json...
Loaded 813 chunks from .rag_index\chunks.json
Ingested 6 documents
Created 810 chunks
System stats: {'indexed': True, 'document_count': 813, 'model': 'gemma4:e4b', 'embedding_model': 'BAAI/bge-large-en-v1.5', 'index_dir': '.rag_index'}


## Query Interface

In [4]:
# Function to query the RAG system
def ask(question: str, top_k: int = 3):
    """Ask a question and display results."""
    result = rag.query(question, top_k=top_k)
    
    print(f"Question: {question}\n")
    print(f"Answer: {result.answer}\n")
    print("Sources:")
    for i, source in enumerate(result.sources, 1):
        print(f"  [{i}] {source['source']}, Page {source['page']} (score: {source['score']:.3f})")
    
    return result

### Example Queries

In [5]:
# Example: Ask about security
ask("What are the main security concerns for AI agents?")

Question: What are the main security concerns for AI agents?

Answer: Main security concerns for AI agents include several new classes of risk beyond typical LLM vulnerabilities [3]. Specifically, risks involve agent privilege escalation, model misuse, hallucination, and data poisoning [3]. Additional concerns unique to agentic systems encompass context poisoning, data leakage, and unauthorized state retention due to persistent memory [1]. Furthermore, autonomous planning can lead to misaligned objectives, unpredictable emergent behavior, recursive decision loops, and general threats like prompt injection and insecure output handling [1], [3].

[Sources: [1], [3]]

Sources:
  [1] Securing-Agentic-Applications-Guide-1.0.pdf, Page 31 (score: 0.029)
  [2] Securing-Agentic-Applications-Guide-1.0.pdf, Page 70 (score: 0.028)
  [3] Cheat-Sheet-Red-Teaming-AI-Solution-Landscape-Q226.pdf, Page 2 (score: 0.028)


RAGQueryResult(answer='Main security concerns for AI agents include several new classes of risk beyond typical LLM vulnerabilities [3]. Specifically, risks involve agent privilege escalation, model misuse, hallucination, and data poisoning [3]. Additional concerns unique to agentic systems encompass context poisoning, data leakage, and unauthorized state retention due to persistent memory [1]. Furthermore, autonomous planning can lead to misaligned objectives, unpredictable emergent behavior, recursive decision loops, and general threats like prompt injection and insecure output handling [1], [3].\n\n[Sources: [1], [3]]', sources=[{'text': 'Page 30 \n \ngenai.owasp.org \n- \n3. Agentic Developer \nGuidelines \nAgentic AI Developer Security Guidelines: A Lifecycle \nApproach \nBuilding secure Agentic AI systems requires integrating security consi...', 'source': 'Securing-Agentic-Applications-Guide-1.0.pdf', 'page': 31, 'section': '3. Agentic Developer', 'score': 0.028782894736842105}, {

In [6]:
# Example: Ask about OWASP Top 10
ask("What does the OWASP Top 10 say about AI security?")

Question: What does the OWASP Top 10 say about AI security?

Answer: The OWASP Top 10 for Agentic Applications (or OWASP Agentic Top 10) is a reference guide designed to help security leaders and practitioners understand and address the top 10 most impactful threats in AI security [1], [2]. It uses a familiar and practical format to offer concise, actionable guidance on these threats, including descriptions, common vulnerabilities, example scenarios, and implementable mitigations [1]. While the full Agentic Security Initiative has comprehensive guidelines covering the entire lifecycle of Agentic applications, the Top 10 serves as a "compass and go-to reference" for starting the process of securing Agentic apps [2]. The Top 10 for Agentic Applications is planned for Version 2026 [3].

Sources:
  [1] OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 7 (score: 0.032)
  [2] OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 7 (score: 0.032)
  [3] OWASP-Top-10-for-Agentic

RAGQueryResult(answer='The OWASP Top 10 for Agentic Applications (or OWASP Agentic Top 10) is a reference guide designed to help security leaders and practitioners understand and address the top 10 most impactful threats in AI security [1], [2]. It uses a familiar and practical format to offer concise, actionable guidance on these threats, including descriptions, common vulnerabilities, example scenarios, and implementable mitigations [1]. While the full Agentic Security Initiative has comprehensive guidelines covering the entire lifecycle of Agentic applications, the Top 10 serves as a "compass and go-to reference" for starting the process of securing Agentic apps [2]. The Top 10 for Agentic Applications is planned for Version 2026 [3].', sources=[{'text': 'ications (or OWASP Agentic Top 10) serves as a compass and go-to \nreference for security leaders and practitioners to understand and address the Top 10 highest-impact \nthreats and begin their journey....', 'source': 'OWASP-Top-10

## Interactive Chat

In [7]:
# Interactive chat function
def chat():
    """Interactive chat loop."""
    print('RAG Chatbot - Type your question or "quit" to exit\n')
    
    while True:
        question = input('You: ')
        if question.lower() in ['quit', 'exit', 'q']:
            print('Goodbye!')
            break
        
        result = rag.query(question)
        print(f"\nAssistant: {result.answer}\n")
        print('-' * 50)

In [8]:
# Start interactive chat (uncomment to use)
# chat()

In [9]:
# RAGAS Evaluation
from src.test_case import EvalCase
from src.evaluator import Evaluator

# Initialize evaluator with RAG system
evaluator = Evaluator(rag)
print("Evaluator initialized")

Evaluator initialized


In [ ]:
# Run evaluation on security basics test cases
from src.test_case import CASES, EvalCase

results = evaluator.run_eval("tests/eval_cases/security_basics.py")
print(f"Evaluated {len(results)} test cases")

Evaluated 6 test cases


In [11]:
# Display evaluation results
evaluator.print_results(results)


EVALUATION RESULTS
Question                                 Faith    Relev    Prec     Recall  
--------------------------------------------------------------------------------
What is prompt injection in LLM appli... 0.66    0.85    0.75    0.32
What does the OWASP Top 10 for LLM sa... 0.49    0.70    0.50    0.38
What is insecure output handling?        0.55    0.55    0.25    0.35
What are the main security concerns f... 0.73    1.00    1.00    0.83
What is agent privilege escalation?      0.77    0.55    0.25    0.50
What is red teaming in the context of... 0.73    1.00    1.00    0.63
--------------------------------------------------------------------------------
AVERAGE                                  0.66    0.77    0.62    0.50


In [12]:
# Inspect cases with low faithfulness scores
print("\n=== Cases with faithfulness < 0.7 ===")
for r in results:
    if r.faithfulness < 0.7:
        print(f"\nQ: {r.question}")
        print(f"A: {r.answer[:200]}...")
        print(f"Ground Truth: {r.ground_truth[:200]}...")
        print(f"Contexts: {r.contexts[:2]}")
        print("---")


=== Cases with faithfulness < 0.7 ===

Q: What is prompt injection in LLM applications?
A: The provided context details specific types of prompt injection rather than a general definition.

Indirect Prompt Injection can occur through two ways:
*   Hidden instruction payloads embedded in web...
Ground Truth: Prompt injection is a vulnerability where attackers inject malicious content into LLM inputs to manipulate model behavior, causing it to follow attacker-provided instructions rather than the original ...
Contexts: ['1. Indirect Prompt Injection via hidden instruction payloads embedded in web pages or documents in a \nRAG scenario silently redirect an agent to exfiltrate sensitive data or misuse connected tools. \n2. Indirect Prompt Injection external communication channels (e.g. email, calendar, teams) sent from \noutside of the company hijacks an agent’s internal communication capability, sending unauthorized \nmessages under a trusted identity.', 'few-shot learning [88]. Together